# Nelson network

Paper figures, drawn from the saved results. All use the non-negativity correction.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../../../"))

import numpy as np

from src import plots as P
from src.run import Experiment, Results
from src.plots import (load_tables, move_corner_label, place_captions, set_font_sizes,
                       set_partition_titles, stack_captions, thin_tick_labels)
from matplotlib.ticker import AutoLocator, ScalarFormatter

from src.fom.astrochemistry import NSEC
# After the imports: src.plots switches matplotlib to Agg.
%matplotlib inline

CASE = "nelson"
OUT_DIR = "."
SWEEP_DIR = "./sweep"
FIG_DIR = "../../../figures/nelson"
POSITIVITY = "on"

## Results

In [ ]:
sweep, rank, hatrom = load_tables(OUT_DIR, POSITIVITY)
# eta = 1e-4 is not shown.
sweep = {s: d[~np.isclose(d["eta"], 1e-4)] for s, d in sweep.items()}
hatrom = {s: d[~np.isclose(d["eta"], 1e-4)] for s, d in hatrom.items()}
taus = sorted(np.unique(sweep["test"]["tau_split"]))
etas = sorted(np.unique(sweep["test"]["eta"]), reverse=True)
qoi = [c.split("|", 1)[1] for c in rank["test"].dtype.names if c.startswith("l2-in-time|")]
n_eval = {s: int(np.max(sweep[s]["n_tracers"])) for s in sweep}

## Errors

In [ ]:
fig_errors, axes = P.sweep_panel_splits(
    sweep["train"], rank["train"], sweep["test"], rank["test"], qoi,
    n_eval_train=n_eval["train"], n_eval_test=n_eval["test"],
    show_tols=tuple(taus), twin_tol=min(taus),
    hatrom_train=hatrom["train"], hatrom_test=hatrom["test"],
    ylim=(5e-5, 4e-1), sharex=False, row_gap=0.05, figsize=(20, 11.5), legend_ncol=6, legend_y=0.95,
)
set_font_sizes(fig_errors)
# Label every other eta.
thin_tick_labels(fig_errors, [axes[0][1], axes[1][1]], {1, 3, 5, 7})
# Stack the failure labels above the curves; at eta = 1e-5 (left edge) start them left of the point.
blue, orange, green = "#1f77b4", "#d9601b", "#2e7d32"
for row in (0, 1):
    axes[row][1].set_ylim(3e-5, 1e0)
    stack_captions(axes[row][1], order=[blue, orange, green])
    move_corner_label(axes[row][1])
    for t in axes[row][1].texts:
        if hasattr(t, "xy") and np.isclose(t.xy[0], 1e-5):
            t.set_ha("left")
            t.set_position((-8, t.get_position()[1]))

## Clusters, ranks and FOM use

In [ ]:
fig_partition, axes_p = P.sweep_partition_panel(
    sweep["test"], SWEEP_DIR, show_tols=tuple(taus), positivity=POSITIVITY, eval_split="test",
    max_eta_exp=12, bar_headroom=1.1,
)
set_font_sizes(fig_partition)
set_partition_titles(fig_partition, axes_p)
for ax in axes_p:
    thin_tick_labels(fig_partition, [ax], {0, 2, 4, 6})

## hAT-ROM error distributions

In [ ]:
fig_violin, axes_v = P.violin_panel(
    SWEEP_DIR, sorted(taus, reverse=True), qoi, etas=etas, positivity=POSITIVITY, eval_split="test",
    prefix="hatrom_", figsize=(5 * len(taus) + 2, 5 * len(qoi)),
)
# Room above the violins for the failure labels; the label over the first violin is moved off the y axis.
for row in axes_v:
    lo, hi = row[0].get_ylim()
    row[0].set_ylim(lo, hi + 0.8)
    for ax in row:
        for t in ax.texts:
            if hasattr(t, "xy") and np.isclose(t.xy[0], 1):
                t.set_ha("left")
                t.set_position((-12, 4))
# 1e-11 and 1e-12 sit next to each other: drop the 1e-12 label.
for ax in axes_v[-1]:
    thin_tick_labels(fig_violin, [ax], {0, 1, 2, 3})

## Abundances along one test tracer

Each species divided by its FOM maximum.

In [ ]:
TAU, ETA, TRACER = 0.01, 1e-5, 417

exp = Experiment(OUT_DIR, "test", n_workers=1, eta=ETA)
cfg, ctx = exp.cfg, exp.ctx
_, params, _ = exp.split_files
t, y, x = exp.trajectories([TRACER])
atrom = exp.load_atrom(TAU, ETA)

d = P.tracer_data(atrom, params[TRACER], x[0], t[0], y[0], ctx.dt_hydro, cfg.atol, cfg.rtol,
                  method=cfg.ode_method, positivity=(True,), time_scale=NSEC)
peak = np.where(d["peak"] > 0, d["peak"], 1.0)[:, None]
for key in ("y_fom", "y_rom", "y_rom_pos"):
    if key in d:
        d[key] = np.asarray(d[key]) / peak
d["peak"] = np.ones_like(d["peak"])

fig_abund, axes_a, _ = P.abundance_panel(d, None, ctx.net.species, ctx.qoi_indices, ensemble=atrom, figsize=(22, 13))
for ax in np.ravel(axes_a):
    if not ax.get_visible():
        continue
    ax.set_yscale("linear")
    ax.yaxis.set_major_locator(AutoLocator())
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.set_ylim(-0.05, 1.12)
    # AT-ROM lines thicker and on top of the FOM markers.
    for line in ax.get_lines():
        if line.get_linestyle() == "-":
            line.set_linewidth(2.6)
            line.set_zorder(6)
for row in np.atleast_2d(axes_a):
    if row[0].get_visible():
        row[0].set_ylabel("Abundance / max")

## Save

In [ ]:
os.makedirs(FIG_DIR, exist_ok=True)
fig_errors.savefig(f"{FIG_DIR}/panel_errors.pdf", dpi=200, bbox_inches="tight")
fig_partition.savefig(f"{FIG_DIR}/panel_partition.pdf", dpi=200, bbox_inches="tight")
fig_violin.savefig(f"{FIG_DIR}/violin_hatrom.pdf", bbox_inches="tight")
fig_abund.savefig(f"{FIG_DIR}/abundance_over_max_{Results.label(TAU, ETA)}_tracer{TRACER}.pdf", dpi=200, bbox_inches="tight")